In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score
import os

In [ ]:
# Setup spark
spark = ( 
    SparkSession.builder 
    .master("local[*]")
    .appName("Spark RAG Maintenance")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
# Load data
COL_NAMES = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    *[f"sensor_{i}" for i in range(1, 22)]
]

def load_cmaps(path: str):
    df = spark.read.csv(path, sep=" ", inferSchema=True)
    df = df.drop("_c26", "_c27")  # Drop empty columns
    return df.toDF(*COL_NAMES)  # Rename columns

train_df = load_cmaps("../data/raw/train_FD001.txt")
test_df = load_cmaps("../data/raw/test_FD001.txt") 

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")
train_df.show(3)

In [ ]:
max_cycles = train_df.groupBy("engine_id").agg(
        F.max("cycle").alias("max_cycle")
)

train_df = train_df.join(max_cycles, on="engine_id")
train_df = train_df.withColumn("RUL", F.col("max_cycle") - F.col("cycle"))
train_df = train_df.withColumn("failure_soon", (F.col("RUL") <= 30).cast("int"))

print("Label distribution (failure soon):")
train_df.groupBy("failure_soon").count().show()

In [ ]:
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]

# 5 cycle rolling window per engine
w5 = Window.partitionBy("engine_id").orderBy("cycle").rowsBetween(-4, 0)

# 1 cycle lag window
w_lag = Window.partitionBy("engine_id").orderBy("cycle")

# For each col add rolling mean, stddev and 1 cycle lag as new features
for col in SENSOR_COLS:
    train_df = (
        train_df
        .withColumn(f"{col}_roll_mean", F.mean(col).over(w5))
        .withColumn(f"{col}_std_std",   F.stddev(col).over(w5))
        .withColumn(f"{col}_lag_1",     F.lag(col, 1).over(w_lag))
    )

# Drop rows with nulls
train_df = train_df.na.drop()

print("After feature engineering:")
print(f"Rows: {train_df.count():,}")
print(f"Columns: {len(train_df.columns)}")
train_df.printSchema()

In [ ]:
# Store as parquet
os.makedirs("../data/processed", exist_ok=True)
train_df.write.mode("overwrite").parquet("../data/processed/train.parquet")
print("Saved processed data to ../data/processed/train.parquet")

verify = spark.read.parquet("../data/processed/train.parquet")
print("Parquet read-back:") 
print(f"Rows: {verify.count():,}")
print(f"Colrs: {len(verify.columns)}")
verify.printSchema()

In [ ]:
# Conver to pandas
pdf = spark.read.parquet("../data/processed/train.parquet").toPandas()
EXCLUDE_COLS = ["engine_id", "cycle", "max_cycle", "RUL", "failure_soon"]
FEATURE_COLS = [col for col in pdf.columns if col not in EXCLUDE_COLS]

X = pdf[FEATURE_COLS]
y_rul = pdf["RUL"].values
y_cls = pdf["failure_soon"].values

X_train, X_val, y_rul_tr, y_rul_val, y_cls_tr, y_cls_val = train_test_split(
    X, y_rul, y_cls, test_size=0.2, random_state=42
)

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Features used: {len(FEATURE_COLS)}")

In [ ]:
# Regression model for RUL
reg_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.6,
    subasmple=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

reg_model.fit(
    X_train, y_rul_tr,
    eval_set=[(X_val, y_rul_val)],
    verbose=50
)

rul_preds = reg_model.predict(X_val)
rul_rmse = np.sqrt(mean_squared_error(y_rul_val, rul_preds))
print(f"RUL Regression RMSE: {rul_rmse:.4f}")

In [ ]:
# Classification model for failure soon
cls_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=(y_cls_tr == 0).sum() / (y_cls_tr == 1).sum(), # Handle class imbalance
    random_state=42,
    n_jobs=-1
)

cls_model.fit(
    X_train, y_cls_tr,
    eval_set=[(X_val, y_cls_val)],
    verbose=50
)

cls_probs = cls_model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_cls_val, cls_probs)
print("Failure Classification ROC-AUC: {auc:.4f}")


In [ ]:
# Save models
os.makedirs("../models", exist_ok=True)
reg_model.save_model("../models/xgb_rul.json")
cls_model.save_model("../models/xgb_failure_cls.json")
print("Models saved to ../models/")